# 05 - 3D spleen segmentation with MONAI DynUNet (MSD Task09_Spleen)

**Run on a hosted GPU — Kaggle (sidebar > Accelerator > GPU T4) or Colab (Runtime > Change
runtime type > T4) — not on the laptop.** This is the Phase 3b deliverable and it is *required by
step 3 of the brief, not optional*: a 3D segmentation model on the Medical Segmentation
Decathlon spleen task, trained with the MONAI stack.

Starting point, stated openly because the build plan requires it: the training recipe follows
MONAI's official tutorials (the *spleen segmentation* pipeline and the *DynUNet* deep-supervision
setup), rewritten onto this repo's own modules — `src/data/ct_volume.py` owns the input contract
(HU window, spacing, ROI, split) and `src/models/unet3d.py` owns the network, the loss, sliding-
window inference and the Dice math. Nothing preprocessing-shaped is re-typed here, so the masks
the model is *trained* on and the masks it is *scored* on cannot drift apart.

The number that matters is the **Dice score** (target roughly 0.80-0.90), reported as a mean over
held-out cases *plus* a per-case table — on 41 volumes a mean flatters one disaster, so
`dice_report` also reports the minimum and the count of zero-Dice cases.

Three honest constraints of the free tier, and the knobs this notebook exposes for them:

| constraint | knob in CONFIG | this Kaggle run |
|---|---|---|
| T4 VRAM (14.5 GiB free) for a 128^3 crop | `roi`, `batch_train`, `sw_batch_size` | 128^3 / 2 / 2 |
| free session can die mid-run | checkpoint saved every validation round | resume on re-run |
| wall-clock | `epochs_total`, `patches_per_epoch` | 90 / 128 (the full schedule is 100-200) |

A `host: local` run is a smoke test of the code and must never be quoted as a result — the host is
recorded inside `spleen_metrics.json` for exactly this reason.


In [ ]:
import os, sys, time, json, platform
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("python", sys.version.split()[0], "|", platform.release())
try:
    import torch
    print("torch", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("device:", torch.cuda.get_device_name(0),
              "| free VRAM:", round(torch.cuda.mem_get_info()[0] / 2**30, 1), "GB")
except ImportError as exc:
    print("torch missing:", exc)


## 0. Dependencies: MONAI + nibabel, offline-first

Kaggle kernels run with Internet **off** by default, so the wheels are *carried by repo.zip* in
`vendor/` (installed with `--no-index`, i.e. purely from local files). With Internet on, or on
Colab, the same cell falls back to PyPI. One install path either way — the import is what the
rest of the notebook actually needs.


In [ ]:
import glob as _glob
import subprocess as _sp


def _have(mod):
    try:
        __import__(mod)
        return True
    except ImportError:
        return False


if not (_have("monai") and _have("nibabel")):
    wheels = sorted(_glob.glob("/kaggle/input/**/*.whl", recursive=True)
                    + _glob.glob("/content/**/*.whl", recursive=True)
                    + _glob.glob("**/vendor/*.whl", recursive=True))
    vendored = [w for w in wheels if "monai" in os.path.basename(w) or "nibabel" in os.path.basename(w)]
    if vendored:
        print("installing from vendored wheels:", vendored)
        # full paths, not basenames: pip resolves a bare filename against the CWD, not --find-links
        try:
            _sp.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps",
                     *vendored], check=True)
        except (_sp.CalledProcessError, OSError) as exc:
            print("pip install failed:", exc)
            print("falling back to importing the wheels directly (both are pure-Python packages)")
            for _w in reversed(vendored):
                sys.path.insert(0, _w)
    else:
        print("installing monai + nibabel from PyPI (needs internet)")
        _sp.run([sys.executable, "-m", "pip", "install", "-q", "monai==1.5.2", "nibabel==5.4.2"], check=True)

import monai, nibabel
print("monai", monai.__version__, "| nibabel", nibabel.__version__)
import torch, numpy as np, pandas as pd


## 1. Host detection, paths, CONFIG

Same policy as notebooks 03/04: the host is read off the filesystem, never from a flag someone
has to remember to flip, and `run_on` is printed and stored in the metrics so a reviewer can tell
which machine produced the numbers. Deliverables go where the host persists them —
`/kaggle/working` (saved as the kernel's Output when you Save Version) or
`MyDrive/aiml02/results` on Colab; big intermediates stay in the scratch dir.


In [ ]:
RUN_ON = "kaggle" if os.path.isdir("/kaggle") else ("colab" if os.path.isdir("/content") else "local")

PATHS = {
    "kaggle": {"scratch": "/tmp/nb05", "deliverables": "/kaggle/working"},
    "colab": {"scratch": "/content/nb05", "deliverables": "/content/drive/MyDrive/aiml02/results"},
    "local": {"scratch": os.path.join(os.getcwd(), "_nb05_scratch"),
              "deliverables": os.path.join(os.getcwd(), "_nb05_scratch", "results")},
}[RUN_ON]

CONFIG = {
    "seed": 7,
    # architecture (src/models/unet3d.py encodes the shape rules and probes them)
    "num_downsample": 5,       # DynUNet stages-1; filters double per stage up to 320
    "num_aux_heads": 2,        # deep supervision: 2 aux heads + the main head
    # the input contract is imported, not restated - src/data/ct_volume.py is the single source
    # of truth for the window, spacing, ROI and class count.
    "roi": [128, 128, 128],    # 192^3 at batch 2 OOM'd the 14.5 GiB T4 on training step 1
    "foreground_ratio": 0.5,   # half centred on a jittered labelled voxel, half uniform over the volume
    "batch_train": 2,
    "patches_per_epoch": 128,  # 64 optimizer steps per epoch
    "lr": 1e-4,
    "weight_decay": 1e-5,
    "epochs_total": 90,        # Kaggle T4 session budget; the full schedule is 100-200
    "val_after_epoch": 10,     # no validation before this - sliding window is the expensive part
    "val_every": 10,
    "n_val_quick": 3,          # val cadence uses 3 cases; the final test uses all 6
    "sw_batch_size": 2,
    "sw_overlap": 0.5,         # half-overlap stops window seams (see predict_volume's docstring)
    "threshold": 0.5,
    "run_on": RUN_ON,
    "scratch": PATHS["scratch"],
    "out_dir": PATHS["deliverables"],
}
os.makedirs(CONFIG["scratch"], exist_ok=True)
os.makedirs(CONFIG["out_dir"], exist_ok=True)
print(json.dumps(CONFIG, indent=1))


## 2. The code and the dataset

The repo archive is hunted, not assumed — the candidates below are the places it can land on
each host. The dataset is MSD Task09_Spleen (~1.1 GB tar, ~2.5 GB extracted): attach it as a
Kaggle input, park it in Drive, or let `tools/get_msd_spleen.py` download it (that one path needs
Internet **on**, once — the URL and the md5 are pinned in `results/msd_spleen_manifest.json` and
the tool refuses to extract a tar whose md5 does not match MONAI's own table).


In [ ]:
import shutil, zipfile

EXTRACT_TO = os.path.join(CONFIG["scratch"], "repo_src")
CODE_MARKER = os.path.join("src", "data", "ct_volume.py")

CANDIDATES = ["/content/repo.zip", "/kaggle/working/repo.zip",
              "/content/drive/MyDrive/repo.zip", "/content/drive/MyDrive/aiml02/repo.zip",
              "/content/drive/MyDrive/Colab Notebooks/repo.zip",
              os.path.join(os.getcwd(), "..", "repo.zip")]
CANDIDATES += sorted(_glob.glob("/kaggle/input/**/repo.zip", recursive=True))
# Kaggle unzips a zip uploaded to a dataset, so the attached code archive can also arrive as a plain tree
# at /kaggle/input/<dataset>/ instead of <dataset>/repo.zip. Look for the marker file that way too.
_MARKER_GLOB = CODE_MARKER.replace(os.sep, "/")
EXTRACTED_INPUT = []
for _dirpath, _dirnames, _files in os.walk("/kaggle/input"):
    if "competitions" in _dirnames:
        _dirnames.remove("competitions")   # 26k RSNA DICOMs live there and no code
    if os.path.exists(os.path.join(_dirpath, CODE_MARKER)):
        EXTRACTED_INPUT.append(_dirpath)
        _dirnames[:] = []
repo_zip = next((p for p in CANDIDATES if os.path.exists(p)), None)

if repo_zip:
    print("using code archive:", repo_zip, f"({os.path.getsize(repo_zip):,} bytes)")
    shutil.rmtree(EXTRACT_TO, ignore_errors=True)
    os.makedirs(EXTRACT_TO, exist_ok=True)
    with zipfile.ZipFile(repo_zip) as zf:
        zf.extractall(EXTRACT_TO)
elif EXTRACTED_INPUT:
    # /kaggle/input is read-only and the cells below write results next to the code they import, so the
    # attached tree is copied rather than imported in place.
    _src_dir = EXTRACTED_INPUT[0].replace("/" + _MARKER_GLOB, "")
    print("Kaggle unzipped the attached code archive - copying", _src_dir, "->", EXTRACT_TO)
    shutil.rmtree(EXTRACT_TO, ignore_errors=True)
    shutil.copytree(_src_dir, EXTRACT_TO)
elif RUN_ON == "local":
    # Walk up from the notebook's directory until the marker file is found, so this works whether the
    # notebook is run from notebooks/ or from the repository root.
    EXTRACT_TO = os.getcwd()
    while EXTRACT_TO != os.path.dirname(EXTRACT_TO) and not os.path.exists(os.path.join(EXTRACT_TO, CODE_MARKER)):
        EXTRACT_TO = os.path.dirname(EXTRACT_TO)
    print("local run - using the working tree:", EXTRACT_TO)
else:
    raise FileNotFoundError("repo.zip not found. Attach it (Add Input > Your Datasets) or drag it "
                            "into the Colab Files panel; candidates were: " + repr(CANDIDATES))

REPO_DIR = EXTRACT_TO
for marker in (CODE_MARKER, os.path.join("src", "models", "unet3d.py"),
               os.path.join("results", "msd_spleen_manifest.json")):
    assert os.path.exists(os.path.join(REPO_DIR, marker)), f"{marker} missing from the archive"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("repo:", REPO_DIR)


In [ ]:
MSD_TAR = "Task09_Spleen.tar"
MSD_DIR_NAME = "Task09_Spleen"

extracted = sorted(p for p in (_glob.glob("/kaggle/input/**/" + MSD_DIR_NAME, recursive=True)
                               + _glob.glob("/content/drive/MyDrive/**/" + MSD_DIR_NAME, recursive=True)
                               + [os.path.join(CONFIG["scratch"], "msd", MSD_DIR_NAME),
                                  os.path.join(REPO_DIR, "data", "raw", "msd", MSD_DIR_NAME)])
                   if os.path.isdir(os.path.join(p, "imagesTr")))
# os.path.exists, not just glob: the repo-relative candidate is always listed, so an unfiltered
# list is truthy on a host where that file does not exist and copyfile would raise FileNotFoundError.
tars = sorted(p for p in (_glob.glob("/kaggle/input/**/" + MSD_TAR, recursive=True)
                          + _glob.glob("/content/drive/MyDrive/**/" + MSD_TAR, recursive=True)
                          + [os.path.join(REPO_DIR, "data", "raw", "msd", MSD_TAR)])
              if os.path.exists(p))

tool = os.path.join(REPO_DIR, "tools", "get_msd_spleen.py")
dest = os.path.join(CONFIG["scratch"], "msd")
if extracted:
    msd_root = extracted[0]
    print("using extracted dataset:", msd_root)
elif tars:
    os.makedirs(dest, exist_ok=True)
    shutil.copyfile(tars[0], os.path.join(dest, MSD_TAR))
    print(f"copied {MSD_TAR} from input, verifying + extracting with the tool")
    _sp.run([sys.executable, tool, "--dest", dest, "--skip-download"], check=True)
    msd_root = os.path.join(dest, MSD_DIR_NAME)
else:
    print("dataset not attached - downloading (Internet must be ON for this cell)")
    _sp.run([sys.executable, tool, "--dest", dest], check=True)
    msd_root = os.path.join(dest, MSD_DIR_NAME)

from src.data.ct_volume import (HU_MIN, HU_MAX, SPACING_MM, ROI, NUM_CLASSES,
                                list_cases, split_cases, load_case, to_tensors,
                                sample_patch, flip_patch)
cases = list_cases(msd_root)
assert len(cases) == 41, f"expected 41 Task09 cases, got {len(cases)}"
print(f"cases: {len(cases)} | HU window [{HU_MIN}, {HU_MAX}] | spacing {SPACING_MM} mm | "
      f"ROI {ROI} | classes {NUM_CLASSES}")


## 3. One case, loaded the way the model will see it

Plan requirement: inspect one volume's shape and spacing and look at three orthogonal slices.
`load_case` does the real pipeline (NIfTI load, RAS orientation, resample to 1.5 mm isotropic —
bilinear for the image, *nearest* for the mask, because interpolating a mask invents voxels that
belong to no organ — and the [-150, 200] HU window), so what is plotted below is exactly what the
network is trained on.


In [ ]:
import matplotlib.pyplot as plt

demo_case = cases[0]
t0 = time.time()
x0, y0, meta0 = load_case(demo_case["image"], demo_case["label"])
print(f"loaded {meta0['case_id']} in {time.time() - t0:.1f} s")
print("shape before resample:", meta0["shape_before_resample"],
      "-> after:", meta0["shape_after_resample"])
print("spacing before (mm):", meta0["spacing_before_mm"],
      "-> after:", meta0["spacing_after_mm"])
print("label values in file:", meta0["label_values"],
      "| spleen voxels:", f"{meta0['label_voxels']:,}",
      f"({100 * meta0['foreground_fraction']:.2f}% of the volume)")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.4))
d, h, w = x0.shape
slices = [(x0[d // 2], y0[d // 2], "axial"),
          (x0[:, h // 2], y0[:, h // 2], "sagittal"),
          (x0[:, :, w // 2], y0[:, :, w // 2], "coronal")]
for ax, (img, mask, name) in zip(axes, slices):
    ax.imshow(img.T, cmap="gray", origin="lower")
    ax.contour(mask.T, levels=[0.5], colors=["#ff3b30"], linewidths=1.2)
    ax.set_title(f"{name} - {meta0['case_id']}")
    ax.axis("off")
fig.suptitle(f"Task09 case as the model sees it: windowed to [{HU_MIN}, {HU_MAX}] HU, "
             f"{SPACING_MM} mm isotropic (mask contour in red)")
plt.show()


## 4. The split, and the foreground-biased patch sampler

The split is *ours*, not MSD's: Task09 publishes labels for all 41 training volumes and keeps the
challenge test images offline, so there is no public test set to inherit — the per-case table in
section 5 is scored on 6 labelled cases held out by `split_cases` (seed 7: 28 train / 7 val /
6 test). One case is one patient, so this split is patient-safe by construction.

Training never sees a whole volume: `sample_patch` draws a 128^3 crop, half of them centred on a
labelled voxel (a spleen is a few percent of an abdominal CT — uniform crops would be mostly
empty and the loss would learn to predict nothing), records whether the crop was padded, and
`flip_patch` augments with flips applied identically to image and mask.


In [ ]:
split = split_cases(cases, n_val=7, n_test=6, seed=CONFIG["seed"])

_case_cache = {}

def load_cached(case):
    """Loading and resampling a whole CT takes seconds, a patch does not; the 28 training volumes stay in RAM."""
    key = case["image"]
    if key not in _case_cache:
        x, y, _meta = load_case(case["image"], case["label"])
        _case_cache[key] = (x, y)
    return _case_cache[key]

def patch_stream(case_list, rng):
    """Infinite (x, y, record) generator: crop a cached case, flip the patch, tensorise it."""
    while True:
        rng.shuffle(case_list)
        for case in case_list:
            x, y = load_cached(case)
            xp, yp, record = sample_patch(x, y, roi=CONFIG["roi"],
                                          foreground_ratio=CONFIG["foreground_ratio"], rng=rng)
            xp, yp = flip_patch(xp, yp, axes=(0, 1, 2), rng=rng)
            xt, yt = to_tensors(xp, yp)
            yield xt, yt, record

# sanity-check the sampler before any training: 64 patches, what did they actually contain?
rng = np.random.default_rng(CONFIG["seed"])
probe = patch_stream(list(split["train"]), rng)
fg, padded, sizes = 0, 0, []
for _ in range(64):
    _x, _y, rec = next(probe)
    fg += int(rec["centred_on_foreground"])
    padded += int(rec["zero_padded"])
    sizes.append(rec["foreground_fraction"])
print(f"64 probe patches: {fg} centred on foreground (target ~50%), {padded} zero-padded")
print(f"foreground fraction per patch: min {min(sizes):.4f} | mean {np.mean(sizes):.4f}")
print(f"patch tensor shape: {tuple(_x.shape)} | label values: {sorted(set(_y.flatten().tolist()))}")


## 5. Training: DynUNet + DiceCE + deep supervision, mixed precision

The model is `build_dynunet` from `src/models/unet3d.py` — MONAI's DynUNet in the MSD baseline
shape (5 downsamples, filters to 320, instance norm, residual blocks), probed at build time so a
stride/filter mismatch fails here instead of at training step 1. The loss is `DiceCELoss` on the
foreground channel; with deep supervision on, a training forward returns the main head plus two
auxiliary heads stacked on a new axis, and `deep_supervision_loss` weights them 1, 0.4, 0.16.
Validation runs `predict_volume` (sliding-window inference, Gaussian-blended 0.5 overlap) — whole
volumes, the same path the test section and the API use.

Budget honesty: this Kaggle run uses `epochs_total=90` at 64 steps/epoch (the measured wall-clock is in `spleen_metrics.json`).
The full schedule in the build plan is 100-200 epochs; 90 is a free-tier budget run, and the
metrics file records the config so the number is never quoted as a longer run.

The first 40-epoch run of this notebook scored 0.329 mean test Dice while the same checkpoint scored
0.87 on a 128^3 crop that contains the spleen: the sampler was drawing every positive crop from a
voxel *inside* the organ and every negative crop from the volume centre, so the net learned a blob
near the middle of the window instead of the organ. `sample_patch` now jitters the positive centre by
up to a quarter of the ROI and draws negatives uniformly over the volume, and the schedule is longer. If a cell is
re-run in the same session, the last checkpoint in `out_dir` is resumed instead of restarting.


In [ ]:
from src.models.unet3d import (build_dynunet, make_loss, deep_supervision_loss,
                               volume_metrics, dice_report, predict_volume, argmask,
                               save_segmentation_checkpoint, load_segmentation_checkpoint)

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu" and RUN_ON != "local":
    raise AssertionError("this notebook needs the T4: Kaggle sidebar > Accelerator > GPU T4, "
                         "or Colab Runtime > Change runtime type > T4")

loss_fn = make_loss()
net = build_dynunet(num_downsample=CONFIG["num_downsample"],
                    num_aux_heads=CONFIG["num_aux_heads"],
                    probe_input=tuple(CONFIG["roi"])).to(device)
optimizer = torch.optim.AdamW(net.parameters(), lr=CONFIG["lr"],
                              weight_decay=CONFIG["weight_decay"])
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
history, best = [], {"dice": -1.0, "path": None, "epoch": -1}
t_start = time.time()


In [ ]:
CKPT_LAST = os.path.join(CONFIG["out_dir"], "spleen_dynunet_last.pt")

def validate_quick():
    """Sliding window on the first n_val_quick val cases; returns mean foreground Dice."""
    rows = []
    for case in split["val"][: CONFIG["n_val_quick"]]:
        x, y, _ = load_case(case["image"], case["label"])
        probs = predict_volume(net, to_tensors(x, y)[0], roi=CONFIG["roi"],
                               sw_batch_size=CONFIG["sw_batch_size"], device=device,
                               overlap=CONFIG["sw_overlap"])
        rows.append(volume_metrics(argmask(probs, CONFIG["threshold"]), y > 0.5))
    return dice_report(rows)["mean_dice"]

def train():
    start_epoch, rng = 0, np.random.default_rng(CONFIG["seed"])
    if os.path.exists(CKPT_LAST):
        payload = load_segmentation_checkpoint(net, CKPT_LAST, map_location=device)
        start_epoch = int(payload.get("epoch") or -1) + 1
        history.extend(payload.get("history") or [])
        rng = np.random.default_rng(CONFIG["seed"] + start_epoch)
        print(f"resuming from {CKPT_LAST} at epoch {start_epoch}")
    stream = patch_stream(list(split["train"]), rng)

    for epoch in range(start_epoch, CONFIG["epochs_total"]):
        net.train(); running, n_steps = 0.0, CONFIG["patches_per_epoch"] // CONFIG["batch_train"]
        for _ in range(n_steps):
            batch = [next(stream) for _ in range(CONFIG["batch_train"])]
            xs = torch.stack([b[0] for b in batch]).to(device)
            ys = torch.stack([b[1] for b in batch]).to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=(device == "cuda")):
                out = net(xs)
                loss = deep_supervision_loss(out, ys, loss_fn)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running += float(loss.detach().cpu())

        row = {"epoch": epoch, "loss": round(running / n_steps, 4),
               "min": round((time.time() - t_start) / 60, 1)}
        if (epoch + 1 >= CONFIG["val_after_epoch"]) and ((epoch + 1) % CONFIG["val_every"] == 0):
            dice = validate_quick()
            row["val_dice"] = round(dice, 4)
            save_segmentation_checkpoint(net, CKPT_LAST, CONFIG, epoch=epoch, history=history)
            if dice > best["dice"]:
                best.update(dice=dice, epoch=epoch)
                torch.save({"state_dict": {k: v.detach().cpu() for k, v in net.state_dict().items()},
                            "config": CONFIG, "epoch": epoch},
                           os.path.join(CONFIG["out_dir"], "spleen_dynunet_best.pt"))
                row["best"] = True
        history.append(row)
        print(json.dumps(row))

train()
print(f"best quick-val Dice {best['dice']:.4f} at epoch {best['epoch']}")


## 6. Test cases: sliding window, per-case Dice, and the failure case

Scored once, on the 6 held-out labelled cases, with the **best** checkpoint — not the last one.
The per-case table carries Dice, both volumes and the centroid distance in mm: a mask can score
Dice 0.9 and still sit on the wrong lobe, and `volume_metrics` reports the two numbers a
radiologist would actually ask for. `dice_report` keeps zero-Dice cases visible instead of
letting the mean absorb them.


In [ ]:
from src.models.unet3d import load_segmentation_checkpoint

CKPT_BEST = os.path.join(CONFIG["out_dir"], "spleen_dynunet_best.pt")
assert os.path.exists(CKPT_BEST), "no best checkpoint - did the training cell run far enough?"
load_segmentation_checkpoint(net, CKPT_BEST, map_location=device)

t0 = time.time()
per_case = []
for case in split["test"]:
    x, y, meta = load_case(case["image"], case["label"])
    probs = predict_volume(net, to_tensors(x, y)[0], roi=CONFIG["roi"],
                           sw_batch_size=CONFIG["sw_batch_size"], device=device,
                           overlap=CONFIG["sw_overlap"])
    m = volume_metrics(argmask(probs, CONFIG["threshold"]), y > 0.5)
    m["case_id"] = meta["case_id"]
    m["shape_after_resample"] = meta["shape_after_resample"]
    per_case.append(m)
    print(f"{meta['case_id']}: Dice {m['dice']:.4f} | pred {m['pred_volume_mm3'] / 1000:,.0f} ml "
          f"vs ref {m['ref_volume_mm3'] / 1000:,.0f} ml | centroid {m['centroid_dist_mm']} mm")

report = dice_report(per_case)
table = pd.DataFrame(per_case).sort_values("dice")[["case_id", "dice", "pred_volume_mm3",
    "ref_volume_mm3", "centroid_dist_mm", "pred_empty"]]
print()
print(json.dumps(report, indent=1))
print(f"mean Dice on the held-out test cases: {report['mean_dice']:.4f} "
      f"(target ~0.80-0.90) | wall time {(time.time() - t0) / 60:.1f} min")
table


## 7. Overlays: best, median and the failure case

The plan asks for the predicted mask overlaid in a distinct colour on three CT slices, one of them
a failure case. The worst-Dice test case above is plotted here with its per-case numbers in the
title — a reviewer should be able to see *what* the model got wrong, not just that it did.


In [ ]:
order = sorted(per_case, key=lambda r: r["dice"])
show = [order[0], order[len(order) // 2], order[-1]]   # failure, median, best
titles = ["FAILURE (lowest Dice)", "median", "best"]

fig, axes = plt.subplots(3, 3, figsize=(13, 12))
for row, (rec, tag) in enumerate(zip(show, titles)):
    case = next(c for c in split["test"] if c["case_id"] == rec["case_id"])
    x, y, _meta = load_case(case["image"], case["label"])
    probs = predict_volume(net, to_tensors(x, y)[0], roi=CONFIG["roi"],
                           sw_batch_size=CONFIG["sw_batch_size"], device=device,
                           overlap=CONFIG["sw_overlap"])
    mask = argmask(probs, CONFIG["threshold"])
    d, h, w = x.shape
    for col, (img, m, ref, name) in enumerate([(x[d // 2], mask[d // 2], (y > 0.5)[d // 2], "axial"),
                                               (x[:, h // 2], mask[:, h // 2], (y > 0.5)[:, h // 2], "sagittal"),
                                               (x[:, :, w // 2], mask[:, :, w // 2], (y > 0.5)[:, :, w // 2], "coronal")]):
        ax = axes[row, col]
        ax.imshow(img.T, cmap="gray", origin="lower")
        # prediction in red, reference contour in cyan - a miss shows as one without the other
        overlay = np.zeros((*m.T.shape, 4)); overlay[m.T] = [1, 0.23, 0.19, 0.45]
        ax.imshow(overlay)
        ax.contour(ref.T, levels=[0.5], colors=["#00e5ff"], linewidths=1.0)
        if row == 0:
            ax.set_title(name)
        ax.set_ylabel(f"{rec['case_id']} - {tag}\nDice {rec['dice']:.3f}", fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])

fig.suptitle(f"Predicted mask (red) vs reference (cyan outline) - mean test Dice "
             f"{report['mean_dice']:.3f}")
png_path = os.path.join(CONFIG["out_dir"], "spleen_overlays.png")
fig.savefig(png_path, dpi=110, bbox_inches="tight")   # save BEFORE show(): show() clears the figure
plt.show()
print("wrote", png_path)


## 8. What was written, and how to read it honestly

`spleen_metrics.json` records the host, the CONFIG, the quick-val history, the test report and
the per-case table, so the numbers in a report can be traced back to the run that produced them.
The checkpoint travels with its CONFIG for the same reason.

Honest reading, in the same spirit as notebooks 03/04:

- **This split is ours.** Task09 has no public test labels; the test table is 6 labelled cases
  held out from the 41, and `imagesTs` stays unused. Say exactly that in the report.
- **The mean is not the story.** Report `min_dice` and `n_zero_dice` next to the mean; a
  segmentation that quietly misses one spleen is worse than a classifier that misses one case.
- **Free-tier tradeoffs are documented, not hidden.** 128^3 crops at 128 patches/epoch for 40
  epochs is a budget run, not a benchmark run; if VRAM forced a smaller crop or fewer epochs,
  say which knob moved. Dice in the 0.80-0.90 band is the target, and MSD baselines reach higher
  with bigger crops and longer schedules.
- **Attribution.** The recipe follows MONAI's official tutorials, rewritten onto this repo's
  modules — that is normal practice and the report should say it, per the build plan.
- **Education only.** Nothing here is a medical device: `host` says which machine ran it, and
  the disclaimer travels in the metrics file.


In [ ]:
import pandas as pd

metrics_path = os.path.join(CONFIG["out_dir"], "spleen_metrics.json")
csv_path = os.path.join(CONFIG["out_dir"], "spleen_per_case.csv")
payload = {
    "task": "MSD Task09_Spleen",
    "model": "DynUNet (MONAI 1.5.2), deep supervision x2, sliding-window inference",
    "host": {"run_on": RUN_ON, "device": device,
             "python": sys.version.split()[0], "torch": torch.__version__,
             "monai": monai.__version__, "minutes": round((time.time() - t_start) / 60, 1)},
    "config": CONFIG,
    "provenance": {"best_checkpoint": CKPT_BEST,
                   "based_on": "MONAI spleen segmentation + DynUNet tutorials, rewritten onto "
                               "src/data/ct_volume.py and src/models/unet3d.py"},
    "test_report": report,
    "per_case": per_case,
    "disclaimer": "Education/research prototype. Not for clinical use.",
}
with open(metrics_path, "w", encoding="utf-8") as fh:
    json.dump(payload, fh, indent=1)
table.to_csv(csv_path, index=False)

print("saved files:")
for p in (metrics_path, csv_path, png_path, CKPT_LAST,
          os.path.join(CONFIG["out_dir"], "spleen_dynunet_best.pt")):
    if os.path.exists(p):
        print(f"  {p} ({os.path.getsize(p):,} bytes)")
print()
print(json.dumps({"host": payload["host"], "test_report": report}, indent=1))
